In [1]:
import dspy
import json
import dspy.evaluate
from dspy.evaluate import Evaluate

In [2]:
lm = dspy.LM('openai/gpt-4', api_key='')
dspy.configure(lm=lm)

In [3]:
cot = dspy.ChainOfThought("question -> answer")

In [4]:
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the answer is based on the provided question."""
    question: str = dspy.InputField(desc="The question that the answer is based on")
    answer: str = dspy.InputField(desc="The answer generated by the model given the question")
    number_of_claims_that_can_be_inferrred: int = dspy.OutputField(desc="The number of claims in the answer that can be inferred from the question")
    total_number_of_claims: int = dspy.OutputField(desc="The total number of claims in the answer")

In [5]:
def evaluate_faithfulness(question: str, answer: str) -> float:
    # Initialize the DSPy predictor with the defined signature
    faithfulness_checker = dspy.ChainOfThought(CheckCitationFaithfulness)
    
    # Perform the faithfulness check
    result = faithfulness_checker(question=question, answer=answer)

    # Calculate the faithfulness score
    faithfulness_score = result.number_of_claims_that_can_be_inferrred / result.total_number_of_claims if result.total_number_of_claims > 0 else 0.0

    return faithfulness_score
    

In [6]:
with open('data_for_dspy.json') as f:
    data = json.load(f)

In [7]:
dataset = []
for d in data:
    question = (d["about_me"] + " " + d["context"] + " " + d["question"])
    answer = (d['response'])
    dataset.append(dspy.Example(question=question, answer=answer).with_inputs("question"))

In [8]:
# split data into training and testing
split = int(len(data)*0.8)
train_data = dataset[:split]
test_data = dataset[split:]

In [9]:
from dspy.evaluate import Evaluate

In [10]:
evaluate = Evaluate(devset=test_data[:], metric=evaluate_faithfulness, num_threads=8, display_progress=True, display_table=False)

In [12]:
evaluate(cot)

Average Metric: 33.13 / 40 (82.8%): 100%|██████████| 40/40 [00:51<00:00,  1.30s/it]  

2025/01/09 00:38:53 INFO dspy.evaluate.evaluate: Average Metric: 33.12579365079365 / 40 (82.8%)


82.81

In [13]:
# Import the optimizer
from dspy.teleprompt import MIPROv2

# Initialize optimizer
teleprompter = MIPROv2(
    metric=evaluate_faithfulness,
    auto="heavy", # Can choose between light, medium, and heavy optimization runs
)

# Optimize program
print(f"Optimizing program with MIPRO...")
optimized_program = teleprompter.compile(
    cot.deepcopy(),
    trainset=train_data,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    requires_permission_to_run=False,
)

# Save optimize program for future use
optimized_program.save(f"mipro_optimized")

2025/01/09 00:39:36 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING HEAVY AUTO RUN SETTINGS:
num_trials: 50
minibatch: True
num_candidates: 38
valset size: 128

2025/01/09 00:39:36 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/01/09 00:39:36 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/01/09 00:39:36 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=38 sets of demonstrations...


Optimizing program with MIPRO...
Bootstrapping set 1/38
Bootstrapping set 2/38
Bootstrapping set 3/38


  0%|          | 0/32 [00:00<?, ?it/s]2025/01/09 00:39:36 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'question': "I'm a 26 year old investor.\nI want to explore opportunities in emerging markets. Eastern European economies attract investments with improving infrastructure and skilled workforce. \nHow do you view the growth potential of Eastern European markets?", 'answer': '\nEastern European markets have great potential for growth. The improving infrastructure and skilled workforce make them attractive to investors. The region is also home to some of the fastest-growing economies in the world, such as Poland and Romania. Investing in stocks and cryptocurrencies in these markets can be a great way to diversify your portfolio and benefit from the potential for high returns. Additionally, investing in emerging markets can help to reduce risk and provide a hedge against global economic downturns.'}) (input_keys={'question'}) with <function evaluate_fait

Average Metric: 16.00 / 20 (80.0%):  16%|█▌        | 20/128 [01:04<09:47,  5.44s/it]

2025/01/09 00:40:56 ERROR dspy.utils.parallelizer: Error processing item Example({'question': "I'm a 35 year old entrepreneur.\nI want to diversify my investments with some exposure to cryptocurrencies. Ethereum's upgrade to Ethereum 2.0 aims to improve scalability and sustainability. \nHow do you view the potential of Ethereum?", 'answer': '\nI believe Ethereum has great potential. Ethereum 2.0 will make it more scalable and sustainable, which will make it more attractive to investors. Ethereum is also the second largest cryptocurrency by market capitalization, so it is a safe bet for diversifying your investments. I would recommend investing in Ethereum as it has the potential to increase in value over time.'}) (input_keys={'question'}): litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4 in organization org-0MCvwnADBdTSKOErFsDis3IG on tokens per min (TPM): Limit 10000, Used 8887, Requested 1848. Please try a

...

2025/01/09 02:08:30 INFO dspy.evaluate.evaluate: Average Metric: 19.53095238095238 / 25 (78.1%)
2025/01/09 02:08:30 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 78.12 on minibatch of size 25 with parameters ['Predictor 0: Instruction 0'].
2025/01/09 02:08:30 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [71.3, 71.0, 68.05, 73.0, 76.4, 78.49, 72.67, 90.2, 83.33, 61.85, 78.16, 87.93, 87.44, 80.84, 82.64, 82.71, 73.05, 81.95, 91.6, 90.0, 95.0, 86.97, 91.87, 91.4, 94.6, 93.2, 89.2, 86.33, 71.33, 91.4, 80.73, 65.4, 94.0, 86.45, 72.97, 75.53, 86.73, 85.87, 89.67, 89.11, 88.78, 91.0, 94.6, 84.05, 75.87, 89.53, 82.11, 90.2, 77.59, 78.12]
2025/01/09 02:08:30 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [0.0, 0.0, 90.37, 91.15, 91.15]
2025/01/09 02:08:30 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 91.15
2025/01/09 02:08:30 INFO dspy.teleprompt.mipro_optimizer_v2: =============================


2025/01/09 02:08:30 INFO dspy.telepro


Average Metric: 99.71 / 112 (89.0%):  87%|████████▋ | 111/128 [01:00<00:09,  1.80it/s]

2025/01/09 02:09:36 ERROR dspy.utils.parallelizer: Error processing item Example({'question': "I'm a 50 year old financial analyst.\nI'm interested in growth investments with a long-term horizon. Electric vehicle companies announce plans for expanded charging infrastructure. \nHow do you view the potential of electric vehicle companies?", 'answer': '\nElectric vehicle companies are a great long-term investment opportunity. The news of expanded charging infrastructure is a positive sign that the industry is growing and that the companies are investing in their future. I would recommend investing in electric vehicle companies as they are likely to experience significant growth in the coming years.'}) (input_keys={'question'}): litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4 in organization org-0MCvwnADBdTSKOErFsDis3IG on tokens per min (TPM): Limit 10000, Used 8939, Requested 2011. Please try again in 5.7s. V

Average Metric: 113.59 / 127 (89.4%): 100%|██████████| 128/128 [01:51<00:00,  1.14it/s]

2025/01/09 02:10:22 INFO dspy.evaluate.evaluate: Average Metric: 113.5904761904762 / 128 (88.7%)
2025/01/09 02:10:22 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [0.0, 0.0, 90.37, 91.15, 91.15, 88.74]
2025/01/09 02:10:22 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 91.15
2025/01/09 02:10:22 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/01/09 02:10:22 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/01/09 02:10:22 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 91.15!


In [16]:
# Evaluate optimized program
print(f"Evaluate optimized program...")
evaluate(optimized_program, devset=test_data[:])

Evaluate optimized program...
Average Metric: 38.00 / 40 (95.0%): 100%|██████████| 40/40 [00:36<00:00,  1.10it/s] 

2025/01/09 02:17:51 INFO dspy.evaluate.evaluate: Average Metric: 38.0 / 40 (95.0%)


95.0

In [17]:
test_data[0]['question']

"I'm a software engineer with a passion for technology and innovation.\nI want to explore investment opportunities in the tech sector. The tech sector is experiencing rapid growth due to advancements in AI, cloud computing, and renewable technologies.\nValuations in the tech industry can be volatile, influenced by market speculation and emerging competition.\nInvestors are increasingly focusing on companies with sustainable and innovative business models. \nWhat are some good investment strategies for the tech industry?"

In [18]:
# Run optimized program
print(f"Running optimized program...")
results = cot(question=test_data[0].question)

Running optimized program...


In [19]:
test_data[0]['question']

"I'm a software engineer with a passion for technology and innovation.\nI want to explore investment opportunities in the tech sector. The tech sector is experiencing rapid growth due to advancements in AI, cloud computing, and renewable technologies.\nValuations in the tech industry can be volatile, influenced by market speculation and emerging competition.\nInvestors are increasingly focusing on companies with sustainable and innovative business models. \nWhat are some good investment strategies for the tech industry?"

In [20]:
results.reasoning

'The question is asking for investment strategies in the tech industry. Given the context, the answer should consider the rapid growth, volatility, and focus on sustainability and innovation in the tech sector.'

In [ ]:
test_data[0]['question'] + results.reasoning + results.answer

In [21]:
results.answer

"1. Diversification: Spread your investments across different tech companies to mitigate risk. This could include a mix of established companies and startups, across different tech sub-sectors like AI, cloud computing, and renewable technologies.\n\n2. Long-term perspective: Given the volatility in the tech sector, a long-term investment perspective can help ride out short-term market fluctuations.\n\n3. Research and due diligence: Understand the company's business model, competitive landscape, and financial health before investing. Look for companies with sustainable and innovative business models.\n\n4. Regular monitoring: Keep track of industry trends, technological advancements, and market speculation that could impact your investments.\n\n5. Invest in tech ETFs: Exchange-Traded Funds (ETFs) can provide exposure to a broad range of tech companies, reducing the risk associated with investing in individual stocks.\n\n6. Consider ESG factors: With increasing focus on sustainability, c